# 02 — Subtopic Classification
Classifies each article into AI subtopics using zero-shot or keyword rules.
Input: `data/raw_articles.csv` → Output: `data/classified_articles.csv`

In [1]:
import pandas as pd

df = pd.read_csv('data/raw_articles.csv', parse_dates=['date'])
print(df.shape)
df.head(2)

(23859, 7)


,id,date,section,headline,body,wordcount,url
0,us-news/2025/nov/13/rejecting-generative-ai-in...,2025-11-13,US news,Rejecting generative AI in healthcare won’t pr...,Eric Reinhart’s essay raises thoughtful concer...,322,https://www.theguardian.com/us-news/2025/nov/1...
1,games/2025/nov/19/pushing-buttons-arc-raiders-...,2025-11-19,Games,How generative AI in Arc Raiders started a scr...,"Arc Raiders is, by all accounts, a late game-o...",1602,https://www.theguardian.com/games/2025/nov/19/...


## Option A — Keyword-based (fast, no GPU needed)

In [2]:
SUBTOPIC_KEYWORDS = {
    'Generative AI':  ['chatgpt', 'gpt-4', 'gemini', 'llm', 'large language model', 'generative', 'midjourney', 'dall-e', 'stable diffusion'],
    'AI Regulation':  ['regulation', 'eu ai act', 'policy', 'governance', 'ban', 'legislation', 'compliance', 'gdpr'],
    'AI Safety':      ['alignment', 'existential risk', 'ai safety', 'superintelligence', 'bias', 'hallucination', 'deepfake'],
    'AI & Jobs':      ['job loss', 'automation', 'unemployment', 'workforce', 'replace workers', 'layoff'],
    'AI in Healthcare': ['diagnosis', 'medical ai', 'drug discovery', 'radiology', 'clinical', 'patient'],
    'AI & Big Tech':  ['openai', 'google deepmind', 'microsoft', 'meta ai', 'anthropic', 'nvidia', 'investment'],
}

def classify_keywords(text: str) -> list:
    text_lower = str(text).lower()
    matched = [topic for topic, kws in SUBTOPIC_KEYWORDS.items()
               if any(kw in text_lower for kw in kws)]
    return matched if matched else ['Other']

# Apply to headline + first 500 chars of body for speed
df['text_sample'] = df['headline'] + ' ' + df['body'].str[:500]
df['subtopics'] = df['text_sample'].apply(classify_keywords)

# Explode so one row per (article, subtopic)
df_exp = df.explode('subtopics').rename(columns={'subtopics': 'subtopic'})
print(df_exp['subtopic'].value_counts())

subtopic
Other               17328
AI Regulation        4442
AI & Big Tech        1001
Generative AI         736
AI in Healthcare      494
AI Safety             309
AI & Jobs             274
Name: count, dtype: int64


## Option B — Zero-shot classification (better accuracy, slower)
Uncomment if you have a GPU or are okay with ~30 min CPU runtime for 10k articles.

In [4]:
from transformers import pipeline

classifier = pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli'
)

LABELS = list(SUBTOPIC_KEYWORDS.keys())

def classify_zeroshot(text):
    result = classifier(text[:512], LABELS, multi_label=True)
    # keep labels with score > 0.4
    return [l for l, s in zip(result['labels'], result['scores']) if s > 0.4] or ['Other']

df['subtopics'] = df['text_sample'].apply(classify_zeroshot)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

KeyboardInterrupt: 

In [3]:
df_exp.to_csv('data/classified_articles.csv', index=False)
print('Saved: data/classified_articles.csv')
df_exp[['date', 'headline', 'subtopic']].head(10)

Saved: data/classified_articles.csv


,date,headline,subtopic
0,2025-11-13,Rejecting generative AI in healthcare won’t pr...,Generative AI
0,2025-11-13,Rejecting generative AI in healthcare won’t pr...,AI in Healthcare
1,2025-11-19,How generative AI in Arc Raiders started a scr...,Generative AI
1,2025-11-19,How generative AI in Arc Raiders started a scr...,AI Regulation
2,2025-06-03,AI pioneer announces non-profit to develop ‘ho...,Other
3,2025-12-12,Trump signs executive order aimed at preventin...,Other
4,2025-09-16,How AI is undermining learning and teaching in...,Generative AI
5,2025-12-09,EU opens investigation into Google’s use of on...,Generative AI
6,2025-11-22,Meet the AI workers who tell their friends and...,Other
7,2025-12-28,Could AI relationships actually be good for us?,Generative AI
